In [5]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import importlib, io
import pandas as pd
import pipeline.bronze, pipeline.convert, pipeline.silver

importlib.reload(pipeline.bronze)
importlib.reload(pipeline.convert)
importlib.reload(pipeline.silver)

from pipeline.convert import get_parquet
from pipeline.silver import map_columns, load_schema

In [6]:
retail_cols = list(pd.read_parquet(io.BytesIO(get_parquet("online_retail_ii"))).columns)
print(retail_cols)
print(list(load_schema()))

['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']
['invoice_id', 'product_code', 'product_name', 'quantity', 'order_timestamp', 'unit_price', 'customer_id', 'country']


In [8]:
# 1. Kontrollkällan: alla 8 kolumner ska mappas rätt, inget flaggas
result = map_columns(retail_cols)

expected = {
    "Invoice": "invoice_id", "StockCode": "product_code",
    "Description": "product_name", "Quantity": "quantity",
    "InvoiceDate": "order_timestamp", "Price": "unit_price",
    "Customer ID": "customer_id", "Country": "country",
}
assert result["mapped"] == expected
assert result["flagged"] == [] and result["missing"] == []

# 2. Schema drift: en omdöpt kolumn och en ny
drifted = [("price_sek" if c == "Price" else c) for c in retail_cols] + ["Discount"]
result = map_columns(drifted)

assert result["flagged"] == ["price_sek", "Discount"]
assert result["missing"] == ["unit_price"]
assert len(result["mapped"]) + len(result["flagged"]) == len(drifted), "En kolumn försvann"

print(result)

{'mapped': {'Invoice': 'invoice_id', 'StockCode': 'product_code', 'Description': 'product_name', 'Quantity': 'quantity', 'InvoiceDate': 'order_timestamp', 'Customer ID': 'customer_id', 'Country': 'country'}, 'flagged': ['price_sek', 'Discount'], 'missing': ['unit_price']}


'flagged': ['price_sek', 'Discount'], 'missing': ['unit_price']}